# Mi Primer Modelo de Machine Learning: Regresión Lineal

### Caso de Uso: Predicción de Eficiencia de Combustible (Dataset Auto MPG)

En este notebook construiremos un modelo de Machine Learning. Para este ejercicio, el objetivo es entender qué características de un vehículo influyen más en su consumo de combustible y cómo predecirlo para nuevos diseños.

El objetivo pedagógico es dominar el flujo de trabajo completo de un proyecto de ML.

## El Flujo de Trabajo en Machine Learning

1. **Definición del Problema**: Predecir MPG.
2. **Adquisición y Carga de Datos**: Obtener los datos crudos.
3. **Análisis Exploratorio de Datos (EDA)**: Entender los datos y buscar patrones.
4. **Preprocesamiento de Datos**: Limpiar y preparar los datos para la computadora.
5. **Entrenamiento del Modelo**: Ajustar el algoritmo usando los datos.
6. **Evaluación del Modelo**: Medir la precisión de las predicciones.
7. **Interpretación**: Entender la influencia de las variables.

## Paso 1: Configuración y Carga de Datos

Usaremos el Dataset Auto MPG. Se requiere `pandas` para manipulación, `matplotlib`/`seaborn` para visualización y `scikit-learn` para el modelado.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

%matplotlib inline

try:
    from sklearn.datasets import fetch_openml
    dataset = fetch_openml(name='autompg', version=1, as_frame=True, parser='auto')
    df = dataset.frame
except:
    url = "https://archive.ics.uci.edu/ml/machine-learning-databases/auto-mpg/auto-mpg.data"
    column_names = ['mpg', 'cylinders', 'displacement', 'horsepower', 'weight',
                    'acceleration', 'model_year', 'origin', 'car_name']
    df = pd.read_csv(url, names=column_names, na_values='?', comment='\t', sep=' ', skipinitialspace=True)

print("Datos cargados exitosamente. Dimensiones:", df.shape)

## Paso 2: Análisis Exploratorio de Datos (EDA)

Revisión de la estructura de la tabla, tipos de datos y estadísticas descriptivas básicas.

In [ ]:
display(df.head())
df.info()
display(df.describe())

### Limpieza de Datos

1. Convertir `horsepower` a formato numérico, forzando errores a NaN.
2. Eliminar las filas con valores nulos.
3. Eliminar variables no numéricas irrelevantes para regresión pura (como `car_name`).

In [ ]:
df_clean = df.copy()
df_clean = df_clean.drop('car_name', axis=1)

if df_clean['horsepower'].dtype == 'object':
    df_clean['horsepower'] = pd.to_numeric(df_clean['horsepower'], errors='coerce')

df_clean = df_clean.dropna()
print(f"Dimensiones después de limpiar: {df_clean.shape}")

### Visualización de Relaciones

Análisis de la covarianza entre peso, caballos de fuerza y la variable objetivo (MPG).

In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(data=df_clean, x='weight', y='mpg')
plt.title('Relación: Peso del Vehículo vs MPG')
plt.grid(True)
plt.show()

plt.figure(figsize=(8, 5))
sns.scatterplot(data=df_clean, x='horsepower', y='mpg', color='red')
plt.title('Relación: Caballos de Fuerza vs MPG')
plt.grid(True)
plt.show()

**Interpretación Visual:** Se observa una correlación negativa clara. A mayor peso o potencia, el MPG disminuye. La linealidad justifica el uso del algoritmo seleccionado.

## Paso 3: Preprocesamiento (Train/Test Split)

División del conjunto de datos para aislar la evaluación del modelo y evitar sobreajuste (overfitting). Proporción 80/20.

In [ ]:
X = df_clean.drop('mpg', axis=1)
y = df_clean['mpg']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"X_train: {X_train.shape}, X_test: {X_test.shape}")

## Paso 4: Entrenamiento del Modelo (Regresión Lineal)

La ecuación fundamental formulada es:

$$h_\theta(x) = \theta_0 + \theta_1x_1 + \theta_2x_2 + ... + \theta_nx_n$$

Donde $\theta$ representa los coeficientes óptimos que minimizan el error respecto a $y$.

<div class="alert alert-warning">
<b>Profundización Analítica (Nivel Posgrado): Estimación de Máxima Verosimilitud (MLE)</b>

Bajo la asunción de errores normales $\mathcal{N}(0, \sigma^2)$, minimizar el Error Cuadrático Medio (MSE) es analíticamente equivalente a maximizar la Función de Verosimilitud logarítmica.

Función de Costo (Mínimos Cuadrados Ordinarios):
$$J(\theta) = \frac{1}{2m} \sum_{i=1}^{m} (h_\theta(x^{(i)}) - y^{(i)})^2$$
</div>

In [ ]:
modelo_reg_lin = LinearRegression()
modelo_reg_lin.fit(X_train, y_train)
print("Modelo ajustado.")

## Paso 5: Evaluación del Modelo

Cuantificación del desempeño utilizando el conjunto de prueba aislado mediante RMSE y $R^2$ Score.

In [ ]:
y_pred_test = modelo_reg_lin.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_pred_test))
r2 = r2_score(y_test, y_pred_test)

print(f"RMSE: {rmse:.2f} MPG")
print(f"R^2 Score: {r2:.3f}")

## Paso 6: Interpretación del Modelo

Extracción de los parámetros aprendidos ($\theta$) para determinar la magnitud direccional de cada característica sobre el MPG.

In [ ]:
df_coef = pd.DataFrame({'Caracteristica': X.columns, 'Coeficiente': modelo_reg_lin.coef_})
df_coef = df_coef.sort_values(by='Coeficiente', ascending=False)
print(f"Intercepto: {modelo_reg_lin.intercept_:.2f}\n")
display(df_coef)

## Conclusión y Siguientes Pasos

El modelo explica la varianza con eficacia base. Iteraciones futuras requieren la aplicación de One-Hot Encoding para variables categóricas (origen) y escalado de características continuas para estandarizar la magnitud de los gradientes.